# Pipeline EHBG-FACS · 01 · Métodos Exactos (Branch & Cut)

**Paradigma 1 — cota de costo óptimo determinista con Gurobi.**

Resuelve el CVRP sobre el **tiempo de viaje nominal** con formulación de flujo no dirigida y branch-and-cut (cortes RCI/DFJ como *lazy constraints*). Las ventanas y retrasos entran como **recurso de 2ª etapa** en la evaluación. Garantiza el óptimo en instancias pequeñas; su intratabilidad aparece al crecer *n*.

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

## Gurobi
`gurobipy` trae una licencia **restringida** (≤2000 variables): la formulación no dirigida cabe para n≤50 (n=50 → 1275 aristas). Para n>~63 usa la **licencia académica gratuita** (`grbgetkey`).

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gurobipy"], check=False)
import gurobipy; print("Gurobi", gurobipy.gurobi.version())

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = proto.instances_per_size   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Resolver
El solver valida cada solución post-resolución (capacidad, cobertura, gap≥0). Si una instancia excede la licencia de Gurobi, redúcela o usa licencia académica.

In [ ]:
from svrplab.solvers.exact_bc import ExactBranchCut
solver = ExactBranchCut(time_limit=120.0, verbose=False)
df = runner.run_solver(solver, "exact-bc", bank, env, proto, verbose=True)
df

## Métricas agregadas y figuras

In [ ]:
agg = metrics.aggregate_by_size(df); display(agg)
import matplotlib.pyplot as plt
# Ruta + convergencia B&C de la primera instancia del menor tamaño
inst = bank[SIZES[0]][0]
sol = solver.solve(inst, num_realizations=proto.realizations)
viz.plot_routes(inst, sol.routes, title=f"exact-bc · n={SIZES[0]}"); plt.show()
viz.plot_convergence(sol.extras.get("convergence_log", []), gap=sol.extras.get("gap"),
                     n=SIZES[0]); plt.show()

**Interpretación.** `exact-bc` minimiza el costo de viaje nominal e ignora las ventanas en el MIP, por lo que suele lograr el menor `E[c]` pero con factibilidad baja bajo la estocasticidad (las ventanas se violan). Es la referencia de costo, no de robustez.